Load dataset

In [40]:
from datasets import load_dataset, concatenate_datasets, Audio, get_dataset_split_names

# 1. Fetch dataset metadata to automatically extract all available configurations (languages/sub-datasets)
configs = get_dataset_split_names("amu-cai/CAMEO")
print(f"Found {len(configs)} subsets: {configs}")

# 2. Load all subsets using a loop (List Comprehension)
print("Loading and merging data... This may take a moment.")
datasets_list = [
    load_dataset("amu-cai/CAMEO", split=f'{conf}[:50%]')
    for conf in configs
]


full_dataset = concatenate_datasets(datasets_list)
print(full_dataset)

full_dataset = full_dataset.class_encode_column("emotion")
print(full_dataset[0])

sampling_rate = 16000
full_dataset = full_dataset.cast_column("audio", Audio(sampling_rate=sampling_rate))
print(full_dataset[0])



Found 13 subsets: ['crema_d', 'cafe', 'emns', 'emozionalmente', 'enterface', 'jl_corpus', 'mesd', 'nemo', 'oreau', 'pavoque', 'ravdess', 'resd', 'subesco']
Loading and merging data... This may take a moment.
Dataset({
    features: ['file_id', 'audio', 'emotion', 'transcription', 'speaker_id', 'gender', 'age', 'dataset', 'language', 'license'],
    num_rows: 20631
})


Casting to class labels: 100%|██████████| 20631/20631 [00:04<00:00, 4396.32 examples/s]

{'file_id': '3ca695d9bc369b82571ed75d367bad2ce33c3d06f8b1acb7bada4744aa9546ac.flac', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7990f5557950>, 'emotion': 0, 'transcription': "Don't forget a jacket.", 'speaker_id': 'crema-d_1001', 'gender': 'male', 'age': '51', 'dataset': 'CREMA-D', 'language': 'English', 'license': 'Open Database License'}
{'file_id': '3ca695d9bc369b82571ed75d367bad2ce33c3d06f8b1acb7bada4744aa9546ac.flac', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7990ecd9f020>, 'emotion': 0, 'transcription': "Don't forget a jacket.", 'speaker_id': 'crema-d_1001', 'gender': 'male', 'age': '51', 'dataset': 'CREMA-D', 'language': 'English', 'license': 'Open Database License'}


In [41]:
print(full_dataset[0])


{'file_id': '3ca695d9bc369b82571ed75d367bad2ce33c3d06f8b1acb7bada4744aa9546ac.flac', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x79909c32fe90>, 'emotion': 0, 'transcription': "Don't forget a jacket.", 'speaker_id': 'crema-d_1001', 'gender': 'male', 'age': '51', 'dataset': 'CREMA-D', 'language': 'English', 'license': 'Open Database License'}


In [45]:
emotion_groups = {
    # Angry
    'anger':         'angry',
    'disgust':       'angry',
    'fear':          'angry',
    'anxiety':       'angry',

    # Happy
    'happiness':     'happy',
    'excitement':    'happy',
    'enthusiasm':    'happy',
    'encouragement': 'happy',

    # Neutral
    'neutral':       'neutral',
    'assertiveness': 'neutral',
    'sarcasm':       'neutral',
    'calm':          'neutral',

    # Sad
    'sadness':       'sad',
    'concern':       'sad',
    'apology':       'sad',
    'surprise':      'sad',
}

# New mappings
labels = ['angry', 'happy', 'neutral', 'sad']
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

print(label2id)
print(id2label)

def remap_emotion(batch):
    original_emotion_name = full_dataset.features["emotion"].int2str(batch["emotion"])
    new_emotion_name = emotion_groups[original_emotion_name]
    batch["emotion"] = label2id[new_emotion_name]
    return batch

full_dataset = full_dataset.map(remap_emotion)

print(full_dataset[0])


{'angry': 0, 'happy': 1, 'neutral': 2, 'sad': 3}
{0: 'angry', 1: 'happy', 2: 'neutral', 3: 'sad'}


Map: 100%|██████████| 20631/20631 [00:06<00:00, 3023.67 examples/s]

{'file_id': '3ca695d9bc369b82571ed75d367bad2ce33c3d06f8b1acb7bada4744aa9546ac.flac', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x7990767dac90>, 'emotion': 0, 'transcription': "Don't forget a jacket.", 'speaker_id': 'crema-d_1001', 'gender': 'male', 'age': '51', 'dataset': 'CREMA-D', 'language': 'English', 'license': 'Open Database License'}


In [46]:
from transformers import SeamlessM4TFeatureExtractor, AutoConfig, AutoModelForAudioClassification

model_id = "facebook/w2v-bert-2.0"

# 1. Load Feature Extractor instead of AutoProcessor
feature_extractor = SeamlessM4TFeatureExtractor.from_pretrained(model_id)

config = AutoConfig.from_pretrained(
    model_id,
    num_labels=len(labels),
    label2id=label2id,
    id2label=id2label,
    finetuning_task="audio-classification"
)

# 3. Load Model
model = AutoModelForAudioClassification.from_pretrained(
    model_id, 
    config=config,
    ignore_mismatched_sizes=True
)

Loading weights: 100%|██████████| 773/773 [00:01<00:00, 664.75it/s]
Wav2Vec2BertForSequenceClassification LOAD REPORT from: facebook/w2v-bert-2.0
Key               | Status  | 
------------------+---------+-
projector.bias    | MISSING | 
classifier.weight | MISSING | 
projector.weight  | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# 4. Prepare Dataset Function
def prepare_dataset(batch):
    audio = batch["audio"]
    inputs = feature_extractor(
        audio["array"],
        sampling_rate=sampling_rate,
        return_tensors="pt"
    )
    batch["input_features"] = inputs.input_features[0]
    batch["labels"] = batch["emotion"] 
    
    return batch

# 5. Map with a smaller writer_batch_size to avoid RAM issues
encoded_dataset = full_dataset.map(
    prepare_dataset,
    remove_columns=["audio"],   # only remove the raw audio, keep 'label'
    # num_proc=4,
    writer_batch_size=100 
)



Map:   4%|▍         | 926/20631 [00:22<12:25, 26.44 examples/s]

In [ ]:
print(encoded_dataset[0])

# Train/validation split
final_splits = encoded_dataset.train_test_split(test_size=0.15, seed=42)

print("\nDataset summary:")
print(f"Total number of samples: {len(encoded_dataset)}")
print(f"Split: {final_splits['train']}")
print(f"Split: {final_splits['test']}")

{'file_id': '3ca695d9bc369b82571ed75d367bad2ce33c3d06f8b1acb7bada4744aa9546ac.flac', 'emotion': 0, 'transcription': "Don't forget a jacket.", 'speaker_id': 'crema-d_1001', 'gender': 'male', 'age': '51', 'dataset': 'CREMA-D', 'language': 'English', 'license': 'Open Database License', 'input_features': [[-11.43387222290039, -11.021231651306152, -10.176437377929688, -9.51826000213623, -9.712156295776367, -10.120458602905273, -10.984991073608398, -10.785009384155273, -9.24512004852295, -8.208727836608887, -7.942660331726074, -8.05948543548584, -8.263891220092773, -7.621932029724121, -6.740867614746094, -6.396284580230713, -6.562347412109375, -6.794302940368652, -6.573392868041992, -6.436171054840088, -6.661494255065918, -7.0610222816467285, -6.768389701843262, -6.826505184173584, -6.91566801071167, -7.1995849609375, -7.157879829406738, -7.112460613250732, -7.333870887756348, -7.18951940536499, -7.258166790008545, -7.157954692840576, -6.80894660949707, -6.80849027633667, -6.650004863739014,

In [24]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorCTCWithPadding:
    feature_extractor: Any
    padding: Union[bool, str] = True

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.feature_extractor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        if "labels" in features[0]:
            batch["labels"] = torch.tensor([f["labels"] for f in features], dtype=torch.long)
            
        return batch

# Initialize the collator
data_collator = DataCollatorCTCWithPadding(feature_extractor=feature_extractor, padding=True)

In [25]:
from transformers import TrainerCallback

class ProgressCallback(TrainerCallback):
    def on_epoch_begin(self, args, state, control, **kwargs):
        print(f"\n{'='*50}")
        print(f"Epoch {int(state.epoch) + 1} / {args.num_train_epochs}")
        print(f"{'='*50}")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        step = state.global_step
        total = state.max_steps
        pct = step / total * 100

        if "loss" in logs:
            print(f"  Step {step}/{total} ({pct:.1f}%) | loss: {logs['loss']:.4f} | lr: {logs.get('learning_rate', 0):.2e}")

        if "eval_accuracy" in logs:
            print(f"  >> Evaluation | accuracy: {logs['eval_accuracy']:.4f} | eval_loss: {logs.get('eval_loss', 0):.4f}")

    def on_epoch_end(self, args, state, control, **kwargs):
        print(f"\n  Epoch {int(state.epoch)} completed.")


In [27]:
import evaluate
import numpy as np
from transformers import TrainingArguments, Trainer

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions = predictions[0]
    
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./w2v-bert-emotion",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5, 
    per_device_train_batch_size=4,      
    per_device_eval_batch_size=4,       
    gradient_accumulation_steps=16,     
    num_train_epochs=5,
    logging_steps=10,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    gradient_checkpointing=True,     
    dataloader_pin_memory=False,   
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_splits['train'],
    eval_dataset=final_splits['test'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[ProgressCallback()],  
)


In [28]:
import gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print(f"Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved()/1e9:.2f} GB")
print(f"Free:      {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved())/1e9:.2f} GB")


Allocated: 2.33 GB
Reserved:  2.34 GB
Free:      14.83 GB


In [29]:
trainer.train()


Epoch 1 / 5


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.170733,0.661290
2,18.556866,1.040606,0.661290
3,18.556866,1.002764,0.661290
4,13.702197,0.986635,0.661290
5,13.324951,0.982264,0.661290



  Epoch 1 completed.
  >> Evaluation | accuracy: 0.6613 | eval_loss: 1.1707


Writing model shards: 100%|██████████| 1/1 [00:08<00:00,  8.25s/it]



Epoch 2 / 5
  Step 10/30 (33.3%) | loss: 18.5569 | lr: 2.10e-05

  Epoch 2 completed.
  >> Evaluation | accuracy: 0.6613 | eval_loss: 1.0406


...


Epoch 4 / 5
  Step 20/30 (66.7%) | loss: 13.7022 | lr: 1.10e-05

  Epoch 4 completed.
  >> Evaluation | accuracy: 0.6613 | eval_loss: 0.9866


Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.53s/it]



Epoch 5 / 5
  Step 30/30 (100.0%) | loss: 13.3250 | lr: 1.00e-06

  Epoch 5 completed.
  >> Evaluation | accuracy: 0.6613 | eval_loss: 0.9823


Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.37s/it]


TrainOutput(global_step=30, training_loss=15.194671630859375, metrics={'train_runtime': 315.7798, 'train_samples_per_second': 5.542, 'train_steps_per_second': 0.095, 'total_flos': 2.4073211171913984e+17, 'train_loss': 15.194671630859375, 'epoch': 5.0})

In [30]:
from sklearn.metrics import classification_report, confusion_matrix

predictions = trainer.predict(final_splits['test'])
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

id2label = {v: k for k, v in label2id.items()}
pred_names = [id2label[p] for p in preds]
label_names = [id2label[l] for l in labels]

print(classification_report(label_names, pred_names))

              precision    recall  f1-score   support

       angry       0.66      1.00      0.80        41
     neutral       0.00      0.00      0.00        10
         sad       0.00      0.00      0.00        11

    accuracy                           0.66        62
   macro avg       0.22      0.33      0.27        62
weighted avg       0.44      0.66      0.53        62



/home/wzelek/git/wav2vec2-emotion-detection/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/wzelek/git/wav2vec2-emotion-detection/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/wzelek/git/wav2vec2-emotion-detection/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavio

In [31]:
# 1. Run final evaluation on the test split
metrics = trainer.evaluate()
print("\n--- Final Evaluation Statistics ---")
for key, value in metrics.items():
    print(f"{key}: {value:.4f}")

# 2. View per-class performance (Optional but highly recommended)
# This helps you see if the model struggles with a specific emotion (e.g., 'sad' vs 'neutral')
predictions = trainer.predict(final_splits["test"])
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

from sklearn.metrics import classification_report
report = classification_report(labels, preds, target_names=id2label.values())
print("\nDetailed Classification Report:")
print(report)

  >> Evaluation | accuracy: 0.6613 | eval_loss: 1.1707


RuntimeError: on_train_begin must be called before on_evaluate

In [39]:
# First, make sure you are logged in
# Run this in a terminal: huggingface-cli login

from huggingface_hub import notebook_login; notebook_login()

repo_name = "wav2vec2-bert-emotion-multilingual-cameo"

# Save locally first as a backup
trainer.save_model("./final_model")
feature_extractor.save_pretrained("./final_model")

# Push to Hugging Face Hub
# This will upload the weights, config, and feature extractor automatically
trainer.push_to_hub(repo_name)

print(f"Model successfully uploaded to: https://huggingface.co/dziobak/{repo_name}")

Writing model shards: 100%|██████████| 1/1 [00:06<00:00,  6.37s/it]
Processing Files (2 / 2): 100%|██████████| 2.33GB / 2.33GB, 9.35MB/s  
New Data Upload: 100%|██████████| 2.33GB / 2.33GB, 9.35MB/s  


Model successfully uploaded to: https://huggingface.co/YOUR_USERNAME/wav2vec2-bert-emotion-multilingual-cameo
